# CS4168 Data Mining Group Project — Spotify Tracks 2026

This notebook follows the structure and methodology of Labs 1–5, applied to the `tracks2026.csv` dataset.

**Before submitting:** add your group's own commentary, interpretation, and any extra experiments you run.

## 1. Setup and Load Data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn import manifold, cluster, set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, MinMaxScaler,
    OneHotEncoder, FunctionTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.model_selection import (
    train_test_split, GridSearchCV,
    StratifiedKFold, KFold,
    cross_validate, cross_val_predict
)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score, adjusted_rand_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    auc, accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn import svm
import pickle

RANDOM_STATE = 42

df = pd.read_csv("tracks2026.csv")
df.head()

## 2. Exploratory Data Analysis (EDA)

Following Lab 1, we begin with a thorough exploration of the dataset before any modelling.

### 2.1 Quick Data Exploration

In [ ]:
# Print first 5 rows of the dataframe
df.head(5)

In [ ]:
# Print last 5 rows of the dataframe
df.tail(5)

In [ ]:
# Print statistical summary for all numerical attributes
df.describe()

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate track_id values:", df["track_id"].duplicated().sum())

### 2.2 Quick Examination of Categorical Attributes

Following Lab 1 Section B.2, we examine frequency tables for the categorical columns.

In [ ]:
# Frequency tables (Lab 1 style)
print("Genres:")
display(df["track_genre"].value_counts())

print("\nExplicit:")
display(df["explicit"].value_counts())

In [ ]:
# Genre distribution bar chart (Lab 1 style)
df["track_genre"].value_counts().plot(kind="bar")
plt.xlabel("Genre")
plt.ylabel("Number of Tracks")
plt.title("Number of Tracks by Genre")
plt.xticks(rotation=45)
plt.show()

### 2.3 Distribution Analysis for Numerical Attributes

Following Lab 1 Section C, we examine histograms and box plots for numerical attributes.

In [ ]:
# Histogram of popularity (Lab 1 style)
df["popularity"].hist(bins=30)
plt.xlabel("Popularity")
plt.ylabel("Number of Tracks")
plt.title("Distribution of Popularity")
plt.show()

In [ ]:
# Boxplot of popularity
df.boxplot(column="popularity")
plt.show()

In [ ]:
# Popularity by genre (Lab 1 style: boxplot grouped by categorical attribute)
df.boxplot(column="popularity", by="track_genre", rot=45, figsize=(10, 5))
plt.title("Box plot of Popularity grouped by Genre")
plt.suptitle("")  # remove automatic title
plt.xlabel("Genre")
plt.ylabel("Popularity")
plt.show()

In [ ]:
# Histograms of all numerical features (Lab 1 style)
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
df[numeric_cols].hist(bins=30, figsize=(16, 12))
plt.suptitle("Distributions of Numerical Features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots of numerical features
df[numeric_cols].boxplot(figsize=(16, 5))
plt.title("Boxplots of Numerical Features")
plt.xticks(rotation=45)
plt.show()

### 2.4 Correlation Heatmap

Following Lab 1, we examine correlations between all numerical attributes.

In [ ]:
# Correlation heatmap (Lab 1 style)
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", figsize=(12, 9))
plt.title("Correlation Heatmap")
plt.show()

print("\nCorrelation with popularity:")
display(df[numeric_cols].corr()["popularity"].sort_values(ascending=False))

In [ ]:
# Scatter plot with regression line (Lab 1 regplot style)
sns.regplot(data=df, x="danceability", y="popularity")
plt.title("Danceability vs Popularity")
plt.show()

In [ ]:
# Pair plot of key numerical features (Lab 1 pairplot style)
sns.pairplot(data=df[["danceability", "energy", "acousticness", "valence", "popularity"]].dropna())
plt.show()

In [ ]:
# Violin plot of popularity by genre (Lab 1 Section D style)
sns.violinplot(data=df, x="track_genre", y="popularity")
plt.title("Violin Plot of Popularity by Genre")
plt.xticks(rotation=45)
plt.show()

### 2.5 Distribution Analysis for Categorical Attributes

Following Lab 1 Section E, we examine the relationship between categorical attributes and popularity using pivot tables and stacked charts.

In [ ]:
# Frequency table for explicit (Lab 1 style)
frequency_table = df["explicit"].value_counts(ascending=True)
print("Frequency Table for Explicit:")
print(frequency_table)

In [ ]:
# Pivot table: mean popularity by explicit flag (Lab 1 pivot table style)
pivot_table_EP = df.pivot_table(values="popularity", index="explicit", aggfunc="mean")
print(pivot_table_EP)

# Plot the frequency table
frequency_table.plot(kind="bar")
plt.xlabel("Explicit")
plt.ylabel("Number of Tracks")
plt.title("Tracks by Explicit Flag")
plt.show()

# Plot pivot table
pivot_table_EP.plot(kind="bar")
plt.xlabel("Explicit")
plt.ylabel("Mean Popularity")
plt.title("Mean Popularity by Explicit Flag")
plt.legend().set_visible(False)
plt.show()

In [ ]:
# Pivot table: mean popularity by genre (Lab 1 style)
pivot_genre = df.pivot_table(values="popularity", index="track_genre", aggfunc="mean")
pivot_genre.sort_values("popularity").plot(kind="bar", legend=False, figsize=(8, 4))
plt.xlabel("Genre")
plt.ylabel("Mean Popularity")
plt.title("Mean Popularity by Genre")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Heatmap: mean popularity by genre and explicit (Lab 1 heatmap style)
pivot_table_GE = df.pivot_table(values="popularity", index="track_genre", columns="explicit", aggfunc="mean")
sns.heatmap(pivot_table_GE, annot=True, fmt=".1f")
plt.title("Mean Popularity by Genre and Explicit Flag")
plt.show()

### 2.6 Data Cleaning

Following Lab 2, we make a copy of the dataset before any transformations and handle identified issues.

In [ ]:
# Keep a copy of the original dataset before any transformation (Lab 2 style)
df_original = df.copy()

# Drop identifier/text columns — unlikely to contain useful information (Lab 2 Section B)
df_clean = df.copy()
df_clean.drop(columns=["track_id", "artists", "album_name", "track_name"], inplace=True)

# Treat unrealistic loudness values as missing (loudness on Spotify is typically <= 0 dB)
df_clean.loc[df_clean["loudness"] > 5, "loudness"] = np.nan

print("Missing values after cleaning:")
display(df_clean.isna().sum())

**Observation:** The dataset contains missing values in several columns. No column has more than 25% missing values, so we do not need to consider dropping whole columns (Lab 2 Section C). Missing values in numerical columns will be handled via median imputation inside scikit-learn pipelines to prevent data leakage.

---

**→ Add your own EDA interpretation here.**  
Discuss: the distribution of popularity, genre balance, which features correlate with popularity, and how these EDA findings inform your preprocessing and modelling decisions.

## 3. Clustering (Descriptive Analytics)

Following Lab 3, we construct clustering pipelines using K-Means and DBSCAN.

### 3.1 Data Preparation for Clustering

In [ ]:
# Assignment requirement: drop track_genre for clustering
cluster_df = df_clean.drop(columns=["track_genre"])

cluster_numeric = cluster_df.select_dtypes(include=np.number).columns.tolist()
cluster_categorical = cluster_df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical columns:", cluster_numeric)
print("Categorical columns:", cluster_categorical)

In [ ]:
# Build preprocessing pipeline for clustering (Lab 3 Pipeline + ColumnTransformer style)
# Apply RobustScaler to loudness (outliers), StandardScaler to others (Lab 3 style)
loudness_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

numeric_no_loudness = [c for c in cluster_numeric if c != "loudness"]

standard_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cluster_preprocess = ColumnTransformer(
    transformers=[
        ("loudness", loudness_pipeline, ["loudness"]),
        ("standard", standard_pipeline, numeric_no_loudness),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cluster_categorical)
    ]
)

X_cluster = cluster_preprocess.fit_transform(cluster_df)

# Keep genre codes for external comparison after clustering (Lab 3 cluster analysis)
genre_codes = pd.factorize(df_clean["track_genre"])[0]

print("Cluster matrix shape:", X_cluster.shape)

### 3.2 K-Means Clustering

Following Lab 3 Section D, we use an elbow plot and silhouette scores to determine a suitable value of k.

In [ ]:
# Evaluate several k values using inertia and silhouette (Lab 3 style)
k_values = range(2, 11)
kmeans_results = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cluster)
    kmeans_results.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(X_cluster, labels),
        "ARI_vs_genre": adjusted_rand_score(genre_codes, labels)
    })

kmeans_results = pd.DataFrame(kmeans_results)
display(kmeans_results)

# Elbow plot (Lab 3 style)
plt.figure(figsize=(7, 4))
plt.plot(kmeans_results["k"], kmeans_results["inertia"], marker="o")
plt.title("K-Means Elbow Plot")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(kmeans_results["k"], kmeans_results["silhouette"], marker="o")
plt.title("K-Means Silhouette Scores")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.show()

In [ ]:
# Compare at least two values of k (specification requirement)
for k in [3, 5]:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cluster)
    print(f"\nK-Means k={k}")
    print("Silhouette:", round(silhouette_score(X_cluster, labels), 3))
    print("ARI vs genre:", round(adjusted_rand_score(genre_codes, labels), 3))
    display(pd.crosstab(labels, df_clean["track_genre"], normalize="index").round(2))

In [ ]:
# Scatterplot function for visualising clusters (Lab 3 Section C)
colors = np.array(["orange", "blue", "lime", "khaki", "pink", "green", "purple"])

def clustering_scatterplot(points, labels, centers, title):
    n_clusters = len(np.unique(labels[labels >= 0]))
    for i in range(n_clusters):
        plt.scatter(points[labels == i, 0], points[labels == i, 1],
                    c=colors[i % colors.size], label=f"cluster {i}", s=15)
    if -1 in labels:  # noise points (DBSCAN)
        plt.scatter(points[labels == -1, 0], points[labels == -1, 1],
                    c="black", label="noise", s=10, alpha=0.3)
    if centers is not None:
        plt.scatter(centers[:, 0], centers[:, 1], c="r", marker="*", s=500)
    plt.title(title)
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("y")

In [ ]:
# Train final K-Means model using a Pipeline (Lab 3 Section D style)
chosen_k = 5  # justify this choice based on elbow plot and silhouette scores above

pipe_km = Pipeline(steps=[
    ("preprocess", cluster_preprocess),
    ("kMeans", KMeans(n_clusters=chosen_k, n_init=10, max_iter=300, random_state=RANDOM_STATE))
])

clustering_model = pipe_km.fit(cluster_df)
preprocessed_data = clustering_model["preprocess"].transform(cluster_df)
kmeans_labels = clustering_model["kMeans"].labels_

In [ ]:
# Append cluster centers and project with MDS (Lab 3 Section E1)
data_and_centers = np.r_[preprocessed_data, clustering_model["kMeans"].cluster_centers_]

XYcoordinates = manifold.MDS(n_components=2, normalized_stress="auto", random_state=RANDOM_STATE).fit_transform(data_and_centers)
print("MDS transformation complete")

plt.figure(figsize=(8, 6))
clustering_scatterplot(
    points=XYcoordinates[:-chosen_k, :],
    labels=kmeans_labels,
    centers=XYcoordinates[-chosen_k:, :],
    title=f"K-Means Clusters (k={chosen_k}) — MDS Projection"
)
plt.show()

In [ ]:
# t-SNE visualisation (Lab 3 Section E2)
XYcoordinates_tsne = manifold.TSNE(n_components=2, random_state=RANDOM_STATE).fit_transform(data_and_centers)
print("t-SNE transformation complete")

plt.figure(figsize=(8, 6))
clustering_scatterplot(
    points=XYcoordinates_tsne[:-chosen_k, :],
    labels=kmeans_labels,
    centers=XYcoordinates_tsne[-chosen_k:, :],
    title=f"K-Means Clusters (k={chosen_k}) — t-SNE Projection"
)
plt.show()

In [ ]:
# Cluster analysis: mean value of each attribute per cluster (Lab 3 Section F)
cluster_df_analysis = df_clean.copy()
cluster_df_analysis["cluster"] = pd.Series(kmeans_labels, index=cluster_df_analysis.index)

display(cluster_df_analysis.head())
display(cluster_df_analysis.tail())

print("\nMean attribute values per cluster:")
display(cluster_df_analysis.groupby("cluster").mean(numeric_only=True).round(3))

print("\nMost common genre per cluster:")
display(cluster_df_analysis.groupby("cluster")["track_genre"].agg(lambda x: x.value_counts().index[0]))

### 3.3 DBSCAN Clustering

In [ ]:
# DBSCAN parameter search (Lab 3 style)
dbscan_results = []
for eps in [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    db = DBSCAN(eps=eps, min_samples=10)
    labels = db.fit_predict(X_cluster)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_ratio = np.mean(labels == -1)
    sil = np.nan
    if n_clusters > 1:
        sil = silhouette_score(X_cluster, labels)
    dbscan_results.append({
        "eps": eps,
        "min_samples": 10,
        "n_clusters": n_clusters,
        "noise_ratio": round(noise_ratio, 3),
        "silhouette": sil
    })

display(pd.DataFrame(dbscan_results))

In [ ]:
# Visualise chosen DBSCAN solution (Lab 3 style)
chosen_eps = 3.5  # choose based on the table above
db_final = DBSCAN(eps=chosen_eps, min_samples=10)
db_labels = db_final.fit_predict(X_cluster)

print("Clusters (excluding noise):", len(set(db_labels)) - (1 if -1 in db_labels else 0))
print("Noise ratio:", round(np.mean(db_labels == -1), 3))

plt.figure(figsize=(8, 6))
clustering_scatterplot(
    points=XYcoordinates[:-chosen_k, :],
    labels=db_labels,
    centers=None,
    title=f"DBSCAN Clusters (eps={chosen_eps}) — MDS Projection"
)
plt.show()

print("\nDBSCAN cluster vs genre cross-tabulation:")
display(pd.crosstab(db_labels, df_clean["track_genre"], normalize="index").round(2))

### 3.4 Clustering Conclusion

**→ Add your own conclusion here following the Lab 3 style.** Discuss:
- Which value of k did you choose and why (elbow plot, silhouette scores)?
- Do clusters reveal meaningful structure — do they align with genres?
- How does K-Means compare to DBSCAN for this dataset?
- What does the cluster mean table tell you about each cluster's musical character?

## 4. Classification — Predicting Popularity Category

Following Lab 4, we train and compare binary classifiers using proper pipeline and cross-validation methodology.

### 4.1 EDA for Classification

In [ ]:
# Create binary target (specification requirement)
class_df = df_clean.dropna(subset=["popularity"]).copy()
median_popularity = class_df["popularity"].median()
class_df["popularity_binary"] = (class_df["popularity"] > median_popularity).astype(int)

print("Median popularity:", median_popularity)
print("\nClass distribution:")
display(class_df["popularity_binary"].value_counts())
display(class_df["popularity_binary"].value_counts(normalize=True).rename("proportion").round(3))

In [ ]:
# Check for missing values and outliers in predictor columns (Lab 4 EDA style)
X_class = class_df.drop(columns=["popularity", "popularity_binary", "track_genre"])
y_class = class_df["popularity_binary"]

print("Missing values in predictors:")
display(X_class.isna().sum())

X_class.select_dtypes(include=np.number).boxplot(figsize=(16, 5))
plt.title("Boxplots of Numerical Predictors (Classification)")
plt.xticks(rotation=45)
plt.show()

### 4.2 Data Preparation for Classification

In [ ]:
# Partition predictors into groups for different preprocessing (Lab 4 style)
class_numeric = X_class.select_dtypes(include=np.number).columns.tolist()
class_categorical = X_class.select_dtypes(exclude=np.number).columns.tolist()

# Columns with outliers get RobustScaler; others get StandardScaler (Lab 4 style)
class_outlier_cols = ["loudness", "duration_ms"]
class_standard_cols = [c for c in class_numeric if c not in class_outlier_cols]

outlier_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

standard_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

class_preprocess = ColumnTransformer(
    transformers=[
        ("outlier_cols", outlier_pipeline, class_outlier_cols),
        ("standard_cols", standard_pipeline, class_standard_cols),
        ("categorical", cat_pipeline, class_categorical)
    ]
)

# Visualise the pipeline (Lab 4 style)
set_config(display="diagram")
class_preprocess

In [ ]:
# Train/test split — stratify to preserve class proportions (Lab 4 style)
X_train, X_test, y_train, y_test = train_test_split(
    X_class, y_class, test_size=0.2, shuffle=True, stratify=y_class, random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]} examples")
print(f"Test set:     {X_test.shape[0]} examples")

### 4.3 Model Training with Cross-Validation

Following Lab 4 Section 3, we tune hyperparameters using 10-fold stratified cross-validation on the training set only.

In [ ]:
# 10-fold stratified CV and scoring metrics (Lab 4 style)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "Accuracy": "accuracy",
    "F1-score": "f1",
    "Precision": "precision",
    "Recall": "recall",
    "ROC_AUC": "roc_auc",
    "AP": "average_precision"
}

In [ ]:
# Model 1: Logistic Regression pipeline (Lab 4 SVM-style pipeline)
pipe_lr = Pipeline(steps=[
    ("preprocess", class_preprocess),
    ("lr", LogisticRegression(max_iter=1000))
])

set_config(display="diagram")
pipe_lr

In [ ]:
param_grid_lr = {
    "lr__C": [0.01, 0.1, 1, 10, 100],
    "lr__penalty": ["l1", "l2"],
    "lr__solver": ["liblinear"]
}

lr_search = GridSearchCV(
    pipe_lr, param_grid_lr,
    n_jobs=-1, cv=cv, scoring=scoring,
    refit="F1-score", return_train_score=False
)

lr_search.fit(X_train, y_train)
print(f"Best CV F1 = {lr_search.best_score_:.3f}")
print("Best parameters:", lr_search.best_params_)

LR_best_model = lr_search.best_estimator_
LR_best_cv_f1 = lr_search.best_score_

In [ ]:
# Model 2: Random Forest pipeline (Lab 4 Random Forest style)
pipe_rf = Pipeline(steps=[
    ("preprocess", class_preprocess),
    ("rf", RandomForestClassifier(random_state=RANDOM_STATE))
])

set_config(display="diagram")
pipe_rf

In [ ]:
param_grid_rf = {
    "rf__n_estimators": [50, 100, 200],
    "rf__max_depth": [4, 8, 12, None]
}

rf_search = GridSearchCV(
    pipe_rf, param_grid_rf,
    n_jobs=-1, cv=cv, scoring=scoring,
    refit="F1-score", return_train_score=False
)

rf_search.fit(X_train, y_train)
print(f"Best CV F1 = {rf_search.best_score_:.3f}")
print("Best parameters:", rf_search.best_params_)

RF_best_model = rf_search.best_estimator_
RF_best_cv_f1 = rf_search.best_score_

### 4.4 Cross-Validated Model Comparison (Training Set Only)

Following Lab 4 Section 4, we compare models using the training set only — the test set is reserved for final evaluation.

In [ ]:
# Compute CV metrics (mean +/- std) for both best models (Lab 4 Section 4.1)
class_models = {
    "Logistic Regression (best by F1)": LR_best_model,
    "Random Forest (best by F1)": RF_best_model
}

cv_results_class = {}

for name, model in class_models.items():
    res = cross_validate(
        model, X_train, y_train,
        cv=cv, scoring=scoring,
        return_train_score=False, n_jobs=-1
    )
    cv_results_class[name] = {m: res[f"test_{m}"] for m in scoring.keys()}

for name, metrics_dict in cv_results_class.items():
    print("\n" + name)
    for m, vals in metrics_dict.items():
        print(f"  {m:9s}: mean={np.mean(vals):.3f}, std={np.std(vals):.3f}")

In [ ]:
# Bar chart of CV metrics with error bars (Lab 4 Section 4.2)
metric_names = list(scoring.keys())
labels = list(cv_results_class.keys())

means = {lab: [np.mean(cv_results_class[lab][m]) for m in metric_names] for lab in labels}
stds  = {lab: [np.std(cv_results_class[lab][m])  for m in metric_names] for lab in labels}

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, means[labels[0]], width, yerr=stds[labels[0]], capsize=4, label=labels[0])
ax.bar(x + width/2, means[labels[1]], width, yerr=stds[labels[1]], capsize=4, label=labels[1])
ax.set_xticks(x)
ax.set_xticklabels(metric_names, rotation=0)
ax.set_ylim(0, 1.1)
ax.set_ylabel("CV score (mean ± std)")
ax.set_title("Cross-validated performance on the training set (10-fold CV)")
ax.legend(loc="lower right")
plt.show()

In [ ]:
# ROC curves from out-of-fold predictions (Lab 4 Section 4.3)
plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")

for name, model in class_models.items():
    oof_proba = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    fpr, tpr, _ = roc_curve(y_train, oof_proba)
    auc_val = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc_val:.2f})")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves from out-of-fold predictions (training set, 10-fold CV)")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Precision-Recall curves from out-of-fold predictions (Lab 4 Section 4.4)
plt.figure(figsize=(7, 5))

for name, model in class_models.items():
    oof_proba = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    prec, rec, _ = precision_recall_curve(y_train, oof_proba)
    ap = average_precision_score(y_train, oof_proba)
    plt.plot(rec, prec, lw=2, label=f"{name} (AP={ap:.2f})")

plt.axhline(y=y_train.mean(), color="red", linestyle="--", label="No skill")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves from out-of-fold predictions (training set, 10-fold CV)")
plt.legend(loc="lower left")
plt.show()

In [ ]:
# Final model selection based on CV F1 (Lab 4 Section 4.5)
if LR_best_cv_f1 >= RF_best_cv_f1:
    best_class_name = "Logistic Regression"
    best_class_model = LR_best_model
    best_class_cv_f1 = LR_best_cv_f1
else:
    best_class_name = "Random Forest"
    best_class_model = RF_best_model
    best_class_cv_f1 = RF_best_cv_f1

print(f"Chosen model based on 10-fold CV F1 on the training set: {best_class_name} (mean CV F1 = {best_class_cv_f1:.3f})")

### 4.5 Final Evaluation on the Test Set

Following Lab 4 Section 5, the test set is used **only once** for final reporting.

In [ ]:
# Evaluation function (Lab 4 Section 5.1)
def evaluate_model(X_eval, y_eval, model):
    probabilities = model.predict_proba(X_eval)[:, 1]
    predicted = model.predict(X_eval)

    results = {
        "confusion_matrix": confusion_matrix(y_eval, predicted),
        "accuracy": accuracy_score(y_eval, predicted),
        "precision": precision_score(y_eval, predicted, zero_division=0),
        "recall": recall_score(y_eval, predicted, zero_division=0),
        "f1": f1_score(y_eval, predicted, zero_division=0)
    }

    fpr, tpr, _ = roc_curve(y_eval, probabilities)
    results["fpr"] = fpr
    results["tpr"] = tpr
    results["auc"] = auc(fpr, tpr)

    prc_precision, prc_recall, _ = precision_recall_curve(y_eval, probabilities)
    results["prc_precision"] = prc_precision
    results["prc_recall"] = prc_recall
    results["ap"] = average_precision_score(y_eval, probabilities)

    return results

test_results = evaluate_model(X_test, y_test, best_class_model)

In [ ]:
# Confusion matrix (Lab 4 Section 5.2)
cm_df = pd.DataFrame(
    test_results["confusion_matrix"],
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)
print("Confusion Matrix (Test Set):")
display(cm_df)

In [ ]:
# Metrics bar chart (Lab 4 Section 5.3)
metrics_arr = np.array([test_results["accuracy"], test_results["precision"],
                        test_results["recall"], test_results["f1"]])
df_metrics = pd.DataFrame({best_class_name: metrics_arr},
                           index=["accuracy", "precision", "recall", "F1-score"])
df_metrics.plot.bar(rot=0, figsize=(5, 3), legend=False, title=best_class_name)
plt.show()

In [ ]:
# ROC curve (Lab 4 Section 5.4)
plt.plot([0, 1], [0, 1], linestyle="--", lw=2, color="r", label="Chance", alpha=0.8)
plt.plot(test_results["fpr"], test_results["tpr"], lw=2,
         label=f"{best_class_name} (AUC = {test_results['auc']:.2f})", alpha=0.8)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve for the chosen classifier")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Precision-Recall curve (Lab 4 Section 5.5)
plt.axhline(y=y_test.mean(), color="red", linestyle="--", label="No skill")
plt.plot(test_results["prc_recall"], test_results["prc_precision"],
         label=f"{best_class_name} (AP = {test_results['ap']:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve for the chosen classifier")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Feature importance (if Random Forest was selected)
if best_class_name == "Random Forest":
    feature_names = best_class_model.named_steps["preprocess"].get_feature_names_out()
    importances = best_class_model.named_steps["rf"].feature_importances_
    importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})\
                      .sort_values("importance", ascending=False)
    display(importance_df.head(15))

    importance_df.head(15).sort_values("importance").plot(
        kind="barh", x="feature", y="importance", legend=False, figsize=(8, 5)
    )
    plt.title("Top 15 Classification Feature Importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

In [ ]:
# Save best classification model (Lab 4 Section 6)
pickle.dump(best_class_model, open("best_classification_model.sav", "wb"))
print("Model saved.")

**→ Add your own classification conclusion here following the Lab 4 style.** Discuss:
- Why F1 was chosen as the model selection metric.
- Which model performed better and why.
- What the ROC/PR curves tell you about the model's discrimination ability.
- Which features were most important and what that means for the task.

## 5. Regression — Predicting Popularity Score

Following Lab 5, we train regression models using the original `popularity` column as the target. A separate copy of the dataset is used — popularity is **not** binarised here.

### 5.1 EDA for Regression

In [ ]:
# Use a separate copy of the dataset with original popularity retained (Lab 5 EDA style)
reg_df = df_clean.dropna(subset=["popularity"]).copy()

reg_df["popularity"].hist(bins=50)
plt.xlabel("Popularity")
plt.ylabel("Number of Tracks")
plt.title("Distribution of Popularity (Regression Target)")
plt.show()

reg_df.boxplot(column=["popularity"])
plt.show()

print("Basic statistics of the target variable:")
display(reg_df["popularity"].describe())

In [ ]:
# Check numerical predictors for outliers (Lab 5 style)
X_reg = reg_df.drop(columns=["popularity", "track_genre"])
y_reg = reg_df["popularity"]

X_reg.select_dtypes(include=np.number).boxplot(figsize=(16, 5))
plt.title("Boxplots of Numerical Predictors (Regression)")
plt.xticks(rotation=45)
plt.show()

print("\nBasic statistics:")
display(X_reg.describe())

### 5.2 Data Preparation for Regression

In [ ]:
# Preprocessing pipeline for regression (Lab 5 ColumnTransformer style)
reg_numeric = X_reg.select_dtypes(include=np.number).columns.tolist()
reg_categorical = X_reg.select_dtypes(exclude=np.number).columns.tolist()

reg_outlier_cols = ["loudness", "duration_ms"]
reg_standard_cols = [c for c in reg_numeric if c not in reg_outlier_cols]

reg_preprocess = ColumnTransformer(
    transformers=[
        ("outlier_cols", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())
        ]), reg_outlier_cols),
        ("standard_cols", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), reg_standard_cols),
        ("categorical", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
         reg_categorical)
    ]
).set_output(transform="pandas")

# Inspect transformed output (Lab 5 style — test only, not used in model training)
X_transformed = reg_preprocess.fit_transform(X_reg)
display(X_transformed.head())
display(X_transformed.tail())

In [ ]:
# Train/test split (Lab 5 style)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, shuffle=True, random_state=RANDOM_STATE
)

print(f"Training set: {Xr_train.shape[0]} examples")
print(f"Test set:     {Xr_test.shape[0]} examples")

### 5.3 Model Training with Cross-Validation

Following Lab 5 Section 3, we tune hyperparameters using 10-fold CV on the training set, with PCA and RFE as dimensionality reduction options.

In [ ]:
# 10-fold CV and scoring for regression (Lab 5 style)
cv10 = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

scoring_reg = {
    "neg_MSE": "neg_mean_squared_error",
    "neg_RMSE": "neg_root_mean_squared_error",
    "neg_MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

In [ ]:
# Model 1: Random Forest with dimensionality reduction (Lab 5 Section 3.1)
pipe_rfr = Pipeline(steps=[
    ("preprocess", reg_preprocess),
    ("reduce_dim", "passthrough"),
    ("ttr", TransformedTargetRegressor(
        regressor=RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

N_FEATURES_OPTIONS = [4, 8, 12]
MAX_DEPTH_OPTIONS = [4, 8, 12]

param_grid_rfr = [
    {
        "reduce_dim": [PCA(iterated_power=7)],
        "reduce_dim__n_components": N_FEATURES_OPTIONS,
        "ttr__regressor__max_depth": MAX_DEPTH_OPTIONS
    },
    {
        "reduce_dim": [RFE(LinearRegression())],
        "reduce_dim__n_features_to_select": N_FEATURES_OPTIONS,
        "ttr__regressor__max_depth": MAX_DEPTH_OPTIONS
    }
]

rfr_search = GridSearchCV(
    pipe_rfr, param_grid_rfr,
    scoring=scoring_reg, n_jobs=-1,
    cv=cv10, refit="R2",
    return_train_score=False
)

rfr_search.fit(Xr_train, yr_train)
print(f"Best CV R² = {rfr_search.best_score_:.3f}")
print("Best parameters:", rfr_search.best_params_)

RF_reg_best_model = rfr_search.best_estimator_
RF_reg_best_cv_r2 = rfr_search.best_score_

In [ ]:
# Model 2: Ridge Regression with dimensionality reduction (Lab 5 Section 3.2)
pipe_ridge = Pipeline(steps=[
    ("preprocess", reg_preprocess),
    ("reduce_dim", "passthrough"),
    ("ttr", TransformedTargetRegressor(
        regressor=Ridge(),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

param_grid_ridge = [
    {
        "reduce_dim": [PCA(iterated_power=7)],
        "reduce_dim__n_components": N_FEATURES_OPTIONS,
        "ttr__regressor__alpha": [0.1, 1.0, 10.0]
    },
    {
        "reduce_dim": [RFE(LinearRegression())],
        "reduce_dim__n_features_to_select": N_FEATURES_OPTIONS,
        "ttr__regressor__alpha": [0.1, 1.0, 10.0]
    }
]

ridge_search = GridSearchCV(
    pipe_ridge, param_grid_ridge,
    scoring=scoring_reg, n_jobs=-1,
    cv=cv10, refit="R2",
    return_train_score=False
)

ridge_search.fit(Xr_train, yr_train)
print(f"Best CV R² = {ridge_search.best_score_:.3f}")
print("Best parameters:", ridge_search.best_params_)

Ridge_best_model = ridge_search.best_estimator_
Ridge_best_cv_r2 = ridge_search.best_score_

### 5.4 Cross-Validated Model Comparison (Training Set Only)

Following Lab 5 Section 4, we compare models using CV metrics on the training set.

In [ ]:
# Compute CV metrics (mean +/- std) for both best regression models (Lab 5 Section 4.1)
reg_models = {
    "Random Forest": RF_reg_best_model,
    "Ridge Regression": Ridge_best_model
}

cv_results_reg = {}
negative_metrics_reg = [k for k in scoring_reg if k.startswith("neg_")]

for name, model in reg_models.items():
    res = cross_validate(
        model, Xr_train, yr_train,
        cv=cv10, scoring=scoring_reg,
        return_train_score=False, n_jobs=-1
    )
    cv_results_reg[name] = {m: res[f"test_{m}"] for m in scoring_reg.keys()}

for name, metrics_dict in cv_results_reg.items():
    print("\n" + name)
    for m, vals in metrics_dict.items():
        if m in negative_metrics_reg:
            vals_to_report = -vals
            metric_name = m.replace("neg_", "")
        else:
            vals_to_report = vals
            metric_name = m
        print(f"  {metric_name:5s}: mean={np.mean(vals_to_report):.3f}, std={np.std(vals_to_report):.3f}")

In [ ]:
# RMSE and MAE bar chart (Lab 5 Section 4.2.1)
model_names_reg = list(reg_models.keys())
error_metrics = ["neg_RMSE", "neg_MAE"]
display_errors = ["RMSE", "MAE"]

x = np.arange(len(error_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
for i, name in enumerate(model_names_reg):
    values = [-np.mean(cv_results_reg[name][m]) for m in error_metrics]
    stds = [np.std(cv_results_reg[name][m]) for m in error_metrics]
    ax.bar(x + (i - 0.5) * width, values, width, yerr=stds, capsize=4, label=name)

ax.set_xticks(x)
ax.set_xticklabels(display_errors)
ax.set_ylabel("Error")
ax.set_title("Cross-validated prediction error")
ax.legend()
plt.show()

In [ ]:
# R² bar chart (Lab 5 Section 4.2.2)
fig, ax = plt.subplots(figsize=(4, 4))
means_r2 = [np.mean(cv_results_reg[name]["R2"]) for name in model_names_reg]
stds_r2 = [np.std(cv_results_reg[name]["R2"]) for name in model_names_reg]
ax.bar(model_names_reg, means_r2, yerr=stds_r2, capsize=4)
ax.set_ylabel("R²")
ax.set_title("Cross-validated R²")
plt.show()

In [ ]:
# Distribution of CV results — boxplots (Lab 5 Section 4.2.3)
rmse_data = [-cv_results_reg[name]["neg_RMSE"] for name in model_names_reg]
mae_data = [-cv_results_reg[name]["neg_MAE"] for name in model_names_reg]
r2_data = [cv_results_reg[name]["R2"] for name in model_names_reg]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].boxplot(rmse_data, tick_labels=model_names_reg)
axes[0].set_title("RMSE across CV folds")
axes[0].set_ylabel("RMSE")
axes[0].grid(axis="y", alpha=0.3)

axes[1].boxplot(mae_data, tick_labels=model_names_reg)
axes[1].set_title("MAE across CV folds")
axes[1].set_ylabel("MAE")
axes[1].grid(axis="y", alpha=0.3)

axes[2].boxplot(r2_data, tick_labels=model_names_reg)
axes[2].set_title("R² across CV folds")
axes[2].set_ylabel("R²")
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Final model selection based on CV R² (Lab 5 Section 4.3)
if RF_reg_best_cv_r2 >= Ridge_best_cv_r2:
    best_reg_name = "Random Forest"
    best_reg_model = RF_reg_best_model
    best_reg_cv_r2 = RF_reg_best_cv_r2
else:
    best_reg_name = "Ridge Regression"
    best_reg_model = Ridge_best_model
    best_reg_cv_r2 = Ridge_best_cv_r2

print(f"Chosen model based on 10-fold CV R² on the training set: {best_reg_name} (mean CV R² = {best_reg_cv_r2:.3f})")

### 5.5 Final Evaluation on the Test Set

Following Lab 5 Section 5, the test set is used **only once** for final reporting.

In [ ]:
# Test set performance (Lab 5 Section 5)
yr_pred = best_reg_model.predict(Xr_test)

metrics_reg = {
    "MSE": mean_squared_error(yr_test, yr_pred),
    "RMSE": np.sqrt(mean_squared_error(yr_test, yr_pred)),
    "MAE": mean_absolute_error(yr_test, yr_pred),
    "R2": r2_score(yr_test, yr_pred)
}

print("\nTest set performance")
for m, val in metrics_reg.items():
    print(f"  {m:5s}: {val:.3f}")

In [ ]:
# Predicted vs actual scatter plot (Lab 5 style)
plt.figure(figsize=(5, 5))
plt.scatter(yr_test, yr_pred, s=15, alpha=0.5)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Predicted vs actual values")
min_val = min(yr_test.min(), yr_pred.min())
max_val = max(yr_test.max(), yr_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.show()

In [ ]:
# Save the best regression model (Lab 5 Section 6)
pickle.dump(best_reg_model, open("best_regression_model.sav", "wb"))
print("Regression model saved.")

**→ Add your own regression conclusion here following the Lab 5 style.** Discuss:
- Why R² was chosen as the model selection metric.
- Which model performed better and why.
- Whether predicting exact popularity is harder than predicting high/low popularity (compare regression vs classification difficulty).
- What the predicted vs actual scatter plot tells you about the model's limitations.